# ACE-Step 1.5 — Gradio UI + LoRA/LoKr training on Google Colab

This notebook installs ACE-Step 1.5, persists models / datasets / training outputs on
Google Drive, and launches the full Gradio interface (generation **and** the LoRA / LoKr
training tabs) behind a public `gradio.live` link.

**Before you start:** `Runtime → Change runtime type → GPU` (A100 or L4 recommended for
training; a T4 works for generation and small LoRAs with gradient checkpointing).

Run the cells from top to bottom. The last cell keeps running as long as the UI is up.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Settings

- `USE_DRIVE`: keep checkpoints, datasets and LoRA outputs on Google Drive so they survive
  the end of the Colab session (strongly recommended for training).
- `UI_PASSWORD`: the `gradio.live` link is public — set a password to protect it.

In [ ]:
#@title Settings { display-mode: "form" }
REPO_URL = "https://github.com/walidb27/ACE-Step-1.5"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/ACE-Step"  #@param {type:"string"}
UI_USERNAME = "admin"  #@param {type:"string"}
UI_PASSWORD = ""  #@param {type:"string"}
UI_LANGUAGE = "en"  #@param ["en", "zh", "ja", "pt", "he"]
PREDOWNLOAD_MODELS = True  #@param {type:"boolean"}

REPO_DIR = "/content/ACE-Step-1.5"

## 3. Mount Google Drive (optional)

In [ ]:
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Persistent storage:", DRIVE_DIR)
else:
    print("Drive disabled: everything will be lost when the session ends.")

## 4. Install ACE-Step

Clones the repository and installs the locked dependencies with `uv` in a Python 3.11
virtual environment (matches the prebuilt `flash-attn` wheel). Takes ~5 minutes.

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

%cd {REPO_DIR}
!uv sync --python 3.11

## 5. Link persistent folders to Google Drive

The UI uses paths relative to the repository (`./checkpoints`, `./lora_output`, ...).
They are replaced by symlinks to Drive. Put your audio datasets in
`<DRIVE_DIR>/datasets` and use `./datasets/<name>` as the dataset path in the UI.

In [ ]:
import os
import shutil

PERSISTENT_DIRS = ["checkpoints", "datasets", "lora_output", "lokr_output", "gradio_outputs"]

def link_to_drive(name: str) -> None:
    local = os.path.join(REPO_DIR, name)
    remote = os.path.join(DRIVE_DIR, name)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.symlink(remote, local)

if USE_DRIVE:
    for name in PERSISTENT_DIRS:
        link_to_drive(name)
        print(f"./{name} -> {os.path.join(DRIVE_DIR, name)}")
else:
    for name in PERSISTENT_DIRS:
        os.makedirs(os.path.join(REPO_DIR, name), exist_ok=True)

## 6. Download the models (optional)

Downloads the default models into `./checkpoints` (on Drive if enabled), so later sessions
start immediately. The UI can also download them on demand.

In [ ]:
%cd {REPO_DIR}
if PREDOWNLOAD_MODELS:
    !uv run acestep-download

## 7. Launch the Gradio UI

Wait for the line `Running on public URL: https://xxxx.gradio.live` and open that link.
In the UI:

1. **Service configuration**: pick the models and click *Initialize Service*.
2. **LoRA Training** tab: dataset path `./datasets/<your_dataset>`, then Scan →
   Auto-label → Save → Preprocess → Train. Outputs go to `./lora_output` (Drive).

See `docs/en/LoRA_Training_Tutorial.md` for the full training workflow.
Stop the cell (■) to shut the UI down.

In [ ]:
import shlex

%cd {REPO_DIR}
auth = ""
if UI_PASSWORD:
    auth = (
        f"--auth-username {shlex.quote(UI_USERNAME)} --auth-password {shlex.quote(UI_PASSWORD)}"
    )
else:
    print("WARNING: no password set, anyone with the link can use your GPU.")

!uv run acestep --share --server-name 0.0.0.0 --port 7860 --language {UI_LANGUAGE} {auth}

## Tips

- **Disconnections**: Colab stops idle sessions. Keep the tab open; with Drive enabled,
  checkpoints saved every *N* epochs survive and can be resumed from
  `./lora_output/checkpoints/epoch_XXX`.
- **Low VRAM (T4, 16 GB)**: enable *gradient checkpointing* and use batch size 1.
- **Download a LoRA**: it is already on Drive in `MyDrive/ACE-Step/lora_output/`.